In [ ]:
# ============================================================
# Experiment 2 – Behavioural Preprocessing
# ============================================================

# Description:
# Preprocessing of behavioural reaction time (RT) data
# for the explicit dual-picture AAT.
#
# Steps:
# 1. Merge session data
# 2. Convert RTs to milliseconds
# 3. Determine task congruence from block order
# 4. Remove incorrect trials
# 5. Remove RTs < 150 ms
# 6. Winsorise upper RT tail
# 7. Export cleaned behavioural dataset
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Parameters
# ============================================================

RT_MIN = 150
RT_WINSOR_SD = 2

SESSION_LENGTH = 88
EXPECTED_TRIALS = 176

participants_to_exclude = [8, 29, 43, 44, 48, 64, 71, 72]

# ============================================================
# Helper Functions
# ============================================================

def load_csv(name, dtype=float):

    return pd.read_csv(
        name,
        header=None
    ).to_numpy(dtype=dtype)


def merge_sessions(arr):

    merged = []

    for p in range(arr.shape[1] // 2):

        merged.append(
            np.concatenate([
                arr[:, 2*p],
                arr[:, 2*p + 1]
            ])
        )

    return merged


def convert_rt_to_ms(rt):

    rt = np.asarray(rt, dtype=float)

    if 0.05 <= np.nanmedian(rt) <= 5:
        return rt * 1000

    return rt.copy()

# ============================================================
# Load Raw Data
# ============================================================

rt_raw = load_csv(
    "Raw_Data/reactionTime.csv"
)

valence_raw = load_csv(
    "Raw_Data/valence.csv"
)

reaction_raw = load_csv(
    "Raw_Data/actualReaction.csv",
    dtype=str
)

block_order_raw = load_csv(
    "Raw_Data/blockOrder.csv",
    dtype=str
)

side_raw = load_csv(
    "Raw_Data/frameSide.csv"
)

# ============================================================
# Merge Sessions
# ============================================================

rt_parts = merge_sessions(rt_raw)

valence_parts = merge_sessions(valence_raw)

reaction_parts = merge_sessions(reaction_raw)

side_parts = merge_sessions(side_raw)

# ============================================================
# Participant-wise Block Order
# ============================================================

block_order_vec = (
    block_order_raw
    .flatten()
    .astype(str)
)

block_order_vec = np.char.strip(block_order_vec)

block_order_parts = []

for p in range(len(block_order_vec) // 2):

    block_order_parts.append([

        block_order_vec[2*p],
        block_order_vec[2*p + 1]
    ])

# ============================================================
# First Pass: RT Cleaning
# ============================================================

participant_data = []

global_rt_pool = []

for pid in range(1, len(rt_parts) + 1):

    if pid in participants_to_exclude:
        continue

    idx = pid - 1

    rt = convert_rt_to_ms(
        rt_parts[idx]
    )

    valence = valence_parts[idx].astype(float)

    action = np.char.lower(
        np.char.strip(
            reaction_parts[idx].astype(str)
        )
    )

    side = side_parts[idx]

    block_s1, block_s2 = block_order_parts[idx]

    intended_mask = np.ones(
        len(rt),
        dtype=bool
    )

    # --------------------------------------------------------
    # Determine Task Congruence
    # --------------------------------------------------------

    congruence = np.full(
        len(rt),
        np.nan,
        dtype=object
    )

    # Session 1

    if block_s1.upper() == "A":

        congruence[:44] = "congruent"
        congruence[44:88] = "incongruent"

    elif block_s1.upper() == "B":

        congruence[:44] = "incongruent"
        congruence[44:88] = "congruent"

    # Session 2

    if block_s2.upper() == "A":

        congruence[88:132] = "congruent"
        congruence[132:176] = "incongruent"

    elif block_s2.upper() == "B":

        congruence[88:132] = "incongruent"
        congruence[132:176] = "congruent"

    # --------------------------------------------------------
    # Participant Response Congruence
    # --------------------------------------------------------

    response_congruence = np.where(

        (
            (action == "pull")
            & (valence == 1)
        )

        |

        (
            (action == "push")
            & (valence == 0)
        ),

        "congruent",
        "incongruent"
    )

    correct = (
        intended_mask
        & (response_congruence == congruence)
    )

    # --------------------------------------------------------
    # Accuracy Filtering
    # --------------------------------------------------------

    rt_clean = rt.copy()

    rt_clean[~correct] = np.nan

    # --------------------------------------------------------
    # RT Threshold
    # --------------------------------------------------------

    rt_fast_flag = (

        intended_mask
        & np.isfinite(rt_clean)
        & (rt_clean < RT_MIN)
    )

    rt_clean[rt_fast_flag] = np.nan

    global_rt_pool.extend(

        rt_clean[
            intended_mask
            & np.isfinite(rt_clean)
        ]
    )

    participant_data.append({

        "pid": pid,

        "rt_clean": rt_clean,

        "correct": correct,

        "side": side,

        "valence": valence,

        "congruence": congruence
    })

# ============================================================
# Global Winsorisation
# ============================================================

global_rt_pool = np.asarray(global_rt_pool)

rt_upper = (

    np.nanmean(global_rt_pool)

    +

    RT_WINSOR_SD
    * np.nanstd(global_rt_pool, ddof=1)
)

# ============================================================
# Final Dataset
# ============================================================

rows = []

for pdata in participant_data:

    pid = pdata["pid"]

    rt_clean = pdata["rt_clean"]

    side = pdata["side"]

    valence = pdata["valence"]

    congruence = pdata["congruence"]

    rt_clean = np.where(
        rt_clean > rt_upper,
        rt_upper,
        rt_clean
    )

    valid_trials = np.isfinite(rt_clean)

    for t in np.where(valid_trials)[0]:

        rows.append({

            "participant": pid,

            "trial": int(t + 1),

            "RT_ms": float(rt_clean[t]),

            "logRT": float(
                np.log10(rt_clean[t])
            ),

            "side": (
                "right"
                if side[t] == 1
                else "left"
            ),

            "valence": (
                "positive"
                if valence[t] == 1
                else "negative"
            ),

            "congruence": congruence[t]
        })

# ============================================================
# Save
# ============================================================

df_beh = pd.DataFrame(rows)

df_beh.to_csv(

    "Processed_Data/Exp2_behavioural_clean.csv",

    index=False
)

print("Behavioural preprocessing complete.")

print(f"Final trials: {len(df_beh)}")